Exercise 2:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split # Helper function for splitting data

# --- Function Definitions (from your code) ---

def polynomial_features(X: np.ndarray, degree: int) -> np.ndarray:
    """ Generates the design matrix Phi including the intercept column of 1s. """
    X = X.reshape(-1, 1)
    powers = np.arange(degree + 1)
    Phi = X ** powers
    return Phi

def simulate_polynomial_data(dtrue: int, w_true: np.ndarray, n: int, sigma2: float, a: float):
    rng = np.random
    X = rng.uniform(0, a, size=n)
    sort_idx = np.argsort(X)
    X = X[sort_idx]
    y_true = np.zeros(n)
    for i in range(dtrue + 1):
        y_true += w_true[i] * X**i
    y = y_true + rng.normal(0, np.sqrt(sigma2), size=n)
    return X, y

def sgd_polynomial_track(Phi_train, y_train, Phi_val, y_val,
                         eta=1e-3, batch_size=20, max_iters=20000,
                         tol=1e-6, reg=None, lam=0.0, mode="best"):
    """
    Takes PRE-SCALED Phi matrices as input and adds L1/L2 regularization options.
    reg : str, 'L1' or 'L2'
    lam : float, regularization strength (lambda)
    """
    rng = np.random
    n_samples, n_features = Phi_train.shape
    w = rng.normal(0, 3, size=n_features)
    train_losses, val_losses = [], []
    best_val_loss = np.inf
    best_w = None
    last_w = w.copy()

    for it in range(1, max_iters + 1):
        # Mini-batch (uses Phi_train directly)
        idx = rng.choice(n_samples, size=batch_size, replace=False)
        Xb, yb = Phi_train[idx], y_train[idx] # Xb is batch of Phi features

        # Gradient of the data loss (MSE part)
        pred = Xb @ w
        error = pred - yb
        grad = (1 / batch_size) * (Xb.T @ error)

        # --- Regularization Gradient Addition ---
        if reg == 'L2':
            # Add L2 regularization gradient term (2 * lambda * w)
            # Note: typically the intercept term (w[0]) is NOT regularized
            grad[1:] += 2 * lam * w[1:]
        elif reg == 'L1':
            # Add L1 regularization gradient term (lambda * sign(w))
            # Note: typically the intercept term (w[0]) is NOT regularized
            grad[1:] += lam * np.sign(w[1:])
        # ----------------------------------------

        # Update
        w_new = w - eta * grad

        # Losses (on full train/val for monitoring)
        train_loss = np.mean((Phi_train @ w_new - y_train) ** 2)
        val_loss   = np.mean((Phi_val   @ w_new - y_val)   ** 2)
        
        # Note: When tracking loss for monitoring/plotting, you usually don't 
        # include the regularization term in the MSE calculation itself.
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if it % 1000 == 0:
            # print((Phi_val   @ w_new - y_val)) # Removed this print for cleaner output
            pass

        # Best model tracking
        if mode == "best" and val_loss < best_val_loss:
            best_val_loss = val_loss
            best_w = w_new.copy()
        last_w = w_new.copy()

        # Convergence
        if np.linalg.norm(w_new - w) < tol:
            print(f"Converged after {it} iterations.")
            break
        
        w = w_new
        
    final_w = best_w if mode == "best" and best_w is not None else last_w
    return final_w, train_losses, val_losses


# --- Main Execution / Assignment Answers ---

# === Q2: Generate Training and Test Data (dtrue >= 5) ===
dtrue = 5
w_true = np.array([1.0, 2.0, -1.5, 0.5, 2, -0.5]) # dtrue+1 coefficients
n_total = 100
sigma2 = 5
a = 1 # Interval [0, 1]

X_full, y_full = simulate_polynomial_data(dtrue, w_true, n_total, sigma2, a)
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.1, random_state=42)

# === Q3: Run SGD and Track Loss (Use high degree to show overfitting) ===
fit_degree = 10 # Intentionally high to demonstrate overfitting
eta = 1e-5     # Suitable learning rate for scaled features
max_iters = 500000


# 1. Generate Raw Features

Phi_train_raw = polynomial_features(X_train, fit_degree)
Phi_test_raw = polynomial_features(X_test, fit_degree)
'''
# 2. SEPARATE AND SCALE (Fixes the intercept issue)
intercept_train = Phi_train_raw[:, 0:1]
features_train = Phi_train_raw[:, 1:]
intercept_test = Phi_test_raw[:, 0:1]
features_test = Phi_test_raw[:, 1:]

scaler = StandardScaler()
features_train_scaled = scaler.fit_transform(features_train)
features_test_scaled = scaler.transform(features_test)

# Recombine the scaled features with the untouched intercept column
Phi_train_scaled = np.hstack((intercept_train, features_train_scaled))
Phi_test_scaled = np.hstack((intercept_test, features_test_scaled))
'''

# 3. Run SGD (using mode="best" for Q3 analysis)
w_fit, train_loss_hist, val_loss_hist = sgd_polynomial_track(
    Phi_train_raw, y_train, Phi_test_raw, y_test,
    eta=eta, batch_size=90, max_iters=max_iters, mode="last") # note: mode="best" tracks min test loss weights

# 4. Predictions for Plotting
x_dense = np.linspace(X_train.min(), X_train.max(), 500).reshape(-1, 1)

# Process dense X data using the exact same scaling logic
'''
Phi_dense_raw = polynomial_features(x_dense, fit_degree)
intercept_dense = Phi_dense_raw[:, 0:1]
features_dense = Phi_dense_raw[:, 1:]
features_dense_scaled = scaler.transform(features_dense)
Phi_dense_scaled = np.hstack((intercept_dense, features_dense_scaled))
'''

# Get final predictions (y is unscaled, so direct dot product works)
y_fit_dense = polynomial_features(x_dense, dtrue) @ w_fit
y_true_dense = polynomial_features(x_dense, dtrue) @ w_true # Ground truth prediction

print(f"\n--- Q3 Results ---")
print("Estimated best weights (first 6):", w_fit[:6])
print("Ground truth weights:", w_true)


# Plotting Q3
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Left: Training/Validation Loss (Shows Overfit regime clearly)
axs[0].plot(train_loss_hist, label='Training Loss', color='blue', lw=2)
axs[0].plot(val_loss_hist, label='Validation Loss (test set)', color='red', lw=2)
axs[0].set_ylabel('MSE Loss (Log Scale)')
axs[0].set_title('Q3: Training and Validation Loss (Overfitting)')
axs[0].legend()
axs[0].grid(True)

# Right: Polynomial Fit (Shows perfect fit on train, oscillation on test)
axs[1].scatter(X_train, y_train, color='gray', alpha=0.3, label='Training data')
axs[1].scatter(X_test, y_test, color='orange', alpha=0.3, label='Validation/test data')
axs[1].plot(x_dense, y_fit_dense, color='blue', lw=2, label=f'Estimated fit d={fit_degree}')
axs[1].plot(x_dense, y_true_dense, color='red', lw=2, linestyle='--', label=f'Ground truth d={dtrue}')
axs[1].set_xlabel('x')
axs[1].set_ylabel('y')
axs[1].set_title('Q3: Polynomial Fit: Overfit Behavior')
axs[1].legend()
axs[1].grid(True)

plt.show()


# === Q4: Vary Model Complexity ===

print(f"\n--- Q4 Results ---")

d_models = [1, 3, 5, 8, 12, 20] # Vary model degree: underfit, well-fit, overfit
train_errors_q4 = []
test_errors_q4 = []

for d_model in d_models:
    # 1. Generate Raw Features for current degree
    Phi_train_q4_raw = polynomial_features(X_train, d_model)
    Phi_test_q4_raw = polynomial_features(X_test, d_model)

    # 2. Scale (Bypassing Intercept)
    scaler_q4 = StandardScaler()
    
    # Check if d_model > 0 before attempting to scale features[:, 1:]
    if d_model >= 1:
        features_train_q4 = Phi_train_q4_raw[:, 1:]
        features_test_q4 = Phi_test_q4_raw[:, 1:]
        features_train_q4_scaled = scaler_q4.fit_transform(features_train_q4)
        features_test_q4_scaled = scaler_q4.transform(features_test_q4)
        Phi_train_q4_scaled = np.hstack((Phi_train_q4_raw[:, 0:1], features_train_q4_scaled))
        Phi_test_q4_scaled = np.hstack((Phi_test_q4_raw[:, 0:1], features_test_q4_scaled))
    else:
        # If d_model=0, only the intercept column exists, no scaling needed
        Phi_train_q4_scaled = Phi_train_q4_raw
        Phi_test_q4_scaled = Phi_test_q4_raw


    # 3. Train the model (using mode="last" for Q4 analysis, as requested)
    w_q4, _, _ = sgd_polynomial_track(
        Phi_train_q4_scaled, y_train, Phi_test_q4_scaled, y_test,
        eta=eta,batch_size=5, max_iters=max_iters, mode="last") 

    # 4. Calculate final MSE for train and test sets
    y_pred_train = Phi_train_q4_scaled @ w_q4
    y_pred_test = Phi_test_q4_scaled @ w_q4
    
    train_mse = np.mean((y_pred_train - y_train)**2)
    test_mse = np.mean((y_pred_test - y_test)**2)
    
    train_errors_q4.append(train_mse)
    test_errors_q4.append(test_mse)
    print(f"d_model={d_model:2d} | Train MSE: {train_mse:.4f} | Test MSE: {test_mse:.4f}")


# Plotting Q4
plt.figure(figsize=(8, 5))
plt.plot(d_models, train_errors_q4, label='Training Loss', marker='o', color='blue')
plt.plot(d_models, test_errors_q4, label='Test Loss', marker='o', color='red')
plt.axvline(x=dtrue, color='gray', linestyle='--', label=f'True Degree (dtrue={dtrue})')
plt.xlabel('Model Degree (d_model)')
plt.ylabel('Mean Squared Error (MSE)')
plt.title('Q4: Bias-Variance Tradeoff (Loss vs Model Complexity)')
plt.legend()
plt.grid(True)
plt.show()

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 11 is different from 6)